In [ ]:
import os
import re
import json
import numpy as np
import torch
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer

import mcot_module
from aggregator import CookingAggregator

# Set root directory
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
print(f"Root directory: {ROOT_DIR}")

# Load environment variables
env_path = os.path.join(ROOT_DIR, "src/scene_captioner/.env")
if os.path.exists(env_path):
    load_dotenv(dotenv_path=env_path)
    hf_key = os.getenv("HUGGINGFACE_KEY")
else:
    raise RuntimeError(f".env file not found at {env_path}")

/Users/eddisonpham/Projects/DynaStride-Dynamic-Stride-Windowing-with-MMCoT-for-Multi-Scene-Captioning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Root directory: /Users/eddisonpham/Projects/DynaStride-Dynamic-Stride-Windowing-with-MMCoT-for-Multi-Scene-Captioning


In [2]:
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
def get_embedding(text):
    vec = semantic_model.encode(text)
    return vec / np.linalg.norm(vec)

def semantic_similarity(a, b):
    if not a or not b:
        return 0.0
    vec_a = get_embedding(a)
    vec_b = get_embedding(b)
    return float(np.dot(vec_a, vec_b))

def adaptive_frame_sampling(frame_numbers, base_stride=10, frame_frequency=None):
    if frame_frequency is not None:
        return frame_numbers[::frame_frequency]
    return frame_numbers[::base_stride]

def sliding_window_captioning_dynamic(
    folder_path, frame_numbers, processor, model, model_id,
    window_size=8, base_stride=8, similarity_threshold=0.70
):
    """
    Generates captions for frames using a sliding window approach with dynamic stride adjustment.
    Stride increases when consecutive captions are highly similar, decreases otherwise.
    """
    max_stride = base_stride * 3
    all_captions = []
    prev_caption = None
    stride = base_stride
    idx = 0
    n_frames = len(frame_numbers)

    while idx < n_frames:
        window = frame_numbers[idx:min(idx + window_size, n_frames)]
        caption = mcot_module.analyze_sequence_by_indexes(
            folder_path,
            window,
            processor,
            model,
            device=model.device
        )

        if caption is None or str(caption).strip().lower() == "none":
            caption = ""

        if prev_caption and semantic_similarity(prev_caption, caption) >= similarity_threshold:
            stride = min(int(stride * 1.5), max_stride)
        else:
            if caption.strip():
                all_captions.append(caption)
                prev_caption = caption
            stride = base_stride

        idx += max(1, stride)

    return all_captions

def process_scene(
    scene_folder: str,
    video_folder: str,
    processor,
    model,
    model_id: str,
    window_size: int = 8,
    base_stride: int = 8,
    similarity_threshold: float = 0.7,
    frame_frequency: int = 5
):
    """
    Process a single scene to generate captions.
    
    Args:
        scene_folder: Name of the scene folder
        video_folder: Path to the video folder
        processor: Model processor
        model: Vision-language model
        model_id: Model identifier
        window_size: Size of the sliding window
        base_stride: Base stride for window movement
        similarity_threshold: Threshold for semantic similarity
        frame_frequency: Frequency for frame sampling
        
    Returns:
        Tuple of (scene_id, captions_list)
    """
    scene_path = os.path.join(video_folder, scene_folder)
    if not os.path.isdir(scene_path):
        raise ValueError(f"Scene path does not exist: {scene_path}")
    
    try:
        frame_numbers = sorted(
            [f.replace("frame_", "").replace(".jpg", "") for f in os.listdir(scene_path) if f.endswith(".jpg")],
            key=lambda x: int(x)
        )
        frame_numbers = [f"{int(f):04d}" for f in frame_numbers]
    except (ValueError, OSError) as e:
        raise ValueError(f"Error reading frames from {scene_path}: {e}")

    sampled_frames = adaptive_frame_sampling(frame_numbers, base_stride, frame_frequency)

    captions = sliding_window_captioning_dynamic(
        scene_path, sampled_frames, processor, model, model_id,
        window_size=window_size, base_stride=base_stride,
        similarity_threshold=similarity_threshold
    )

    try:
        scene_id = int(scene_folder.split("_")[1])
    except (IndexError, ValueError) as e:
        raise ValueError(f"Error extracting scene_id from {scene_folder}: {e}")

    return scene_id, captions

def process_video(
    video_id, video_folder, processor, model, model_id,
    window_size=10, base_stride=10, similarity_threshold=0.70, frame_frequency=10
):
    scenes = []
    for d in os.listdir(video_folder):
        print(f"video_folder: {video_folder}, d: {d}")
        if os.path.isdir(os.path.join(video_folder, d)):
            scenes.append(d)
    output = {video_id: {}}
    for scene_folder in tqdm(scenes, desc=f"Processing scenes for {video_id}"):
        print(f"scene_folder: {scene_folder}")
        scene_id, captions = process_scene(
            scene_folder, video_folder, processor, model, model_id,
            window_size, base_stride, similarity_threshold, frame_frequency
        )
        output[video_id][scene_id] = captions
    return output

def get_ground_truth(annotations_file, video_id, scene_count):
    with open(annotations_file, "r") as f:
        annotations_data = json.load(f)

    if video_id not in annotations_data["database"]:
        print(f"Warning: Video {video_id} not found in annotations.")
        return {i: "" for i in range(scene_count)}

    video_annotations = annotations_data["database"][video_id]["annotations"]
    ground_truth_dict = {}
    for i in range(scene_count):
        ground_truth_dict[i] = video_annotations[i]["sentence"] if i < len(video_annotations) else ""
    return ground_truth_dict

def discover_and_create_sampled_file(base_folder, sampled_file_path):
    """
    Automatically discovers videos by scanning base_folder/validation for numbered folders
    and extracting video_ids that exist as subdirectories.
    Creates the sampled_file_path if it doesn't exist.
    """
    if os.path.exists(sampled_file_path):
        print(f"Sampled file already exists: {sampled_file_path}")
        return
    
    validation_path = os.path.join(base_folder, "validation")
    if not os.path.isdir(validation_path):
        raise ValueError(f"Validation folder not found: {validation_path}")
    
    all_videos = []
    for id_folder in os.listdir(validation_path):
        id_path = os.path.join(validation_path, id_folder)
        if not os.path.isdir(id_path):
            continue
        
        # Check for video_id subdirectories in this id folder
        for video_id in os.listdir(id_path):
            video_path = os.path.join(id_path, video_id)
            if os.path.isdir(video_path):
                all_videos.append(f"{id_folder}/{video_id}")
    
    # Sort and write to file
    all_videos = sorted(all_videos)
    
    # Ensure the directory exists
    os.makedirs(os.path.dirname(sampled_file_path), exist_ok=True)
    
    with open(sampled_file_path, "w") as f:
        for video in all_videos:
            f.write(f"{video}\n")
    
    print(f"Created sampled file with {len(all_videos)} videos: {sampled_file_path}")

def initialize_models(aggregator_model_name, cot_model_name, agg_gpu, cot_gpu):
    cot_device = torch.device(f"cuda:{cot_gpu}") if torch.cuda.is_available() else torch.device("cpu")
    agg_device = torch.device(f"cuda:{agg_gpu}") if torch.cuda.is_available() else torch.device("cpu")
    aggregator = CookingAggregator(model_id=aggregator_model_name, device=agg_device)
    processor, model = mcot_module.load_model(model_id=cot_model_name, device=cot_device)
    return aggregator, processor, model

def run_pipeline(base_folder, sampled_file, output_folder, annotations_file,
                 aggregator, processor, model, COT_MODEL_ID):

    os.makedirs(output_folder, exist_ok=True)
    output_path = os.path.join(output_folder, "validation_results.json")

    final_output = {}
    try:
        with open(output_path, "r") as f:
            final_output = json.load(f)
        print(f"Loaded existing JSON with {len(final_output)} videos.")
    except (FileNotFoundError, json.JSONDecodeError) as error:
        print(f"Could not load JSON {output_path}: {error.__class__.__name__}: {error}")
        raise

    try:
        with open(sampled_file, "r") as f:
            all_videos = [line.strip() for line in f if line.strip()]
        print(f"Found {len(all_videos)} videos to process.")
    except FileNotFoundError:
        print(f"Error: Sampled file not found: {sampled_file}")
        raise

    for idx_video, video_rel_path in enumerate(tqdm(all_videos, desc="Processing sampled videos")):
        print(f"\n--- [{idx_video+1}/{len(all_videos)}] Processing video path: {video_rel_path}")
        match = re.match(r"^([^/]+)/([^/]+)$", video_rel_path)
        if not match:
            print(f"Warning: Invalid video path format: {video_rel_path}. Expected 'id/video_id'")
            continue
        video_id = match.group(2)

        video_folder = os.path.join(base_folder, "validation", video_rel_path)

        if video_id in final_output:
            print(f"Info: Video {video_id} already processed; skipping.")
            continue
        if not os.path.isdir(video_folder):
            print(f"Warning: Video folder does not exist: {video_folder}. Skipping.")
            continue

        print(f"Processing frames in folder: {video_folder}")
        try:
            video_result = process_video(
                video_id, video_folder, processor, model, COT_MODEL_ID,
                window_size=10, base_stride=10, similarity_threshold=0.50, frame_frequency=10
            )
            print(f"Video {video_id}: Scene processing complete.")
        except Exception as e:
            print(f"Error: Error processing video {video_id}: {e}")
            continue

        aggregated_scenes = {}
        scene_ids_sorted = sorted(video_result[video_id].keys())
        print(f"Video {video_id}: Found {len(scene_ids_sorted)} scenes to aggregate.")
        for idx, scene_id in enumerate(scene_ids_sorted):
            frame_captions = video_result[video_id][scene_id]
            try:
                aggregated_scenes[idx] = aggregator.generate_cooking_summary(frame_captions) if frame_captions else ""
                print(f"Video {video_id}: Aggregated scene {idx}.")
            except Exception as e:
                print(f"Error: Error aggregating scenes for video {video_id}, scene {idx}: {e}")
                aggregated_scenes[idx] = ""

        scene_count = len(aggregated_scenes)
        ground_truth_dict = get_ground_truth(annotations_file, video_id, scene_count)

        video_data = {
            idx: {
                "ground_truth": ground_truth_dict.get(idx, ""),
                "predicted": aggregated_scenes.get(idx, "")
            }
            for idx in range(scene_count)
        }

        final_output[video_id] = video_data
        try:
            with open(output_path, "w") as f:
                json.dump(final_output, f, indent=2)
            print(f"JSON updated for video {video_id} ({scene_count} scenes).")
        except Exception as e:
            print(f"Error: Failed to save JSON for {video_id}: {e}")
            continue

        del video_result
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Processed and updated JSON for video: {video_id}")


    print(f"Pipeline completed. All results saved to {output_path}")

In [ ]:
AGGREGATOR_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
COT_MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
COT_GPU = 0
AGG_GPU = 1
BASE_FOLDER = os.path.join(ROOT_DIR, "data/YouCookII/raw_videos")
SAMPLED_FILE = os.path.join(ROOT_DIR, "constants/sampled_videos.txt")
OUTPUT_FOLDER = os.path.join(ROOT_DIR, "results/dynastride_results/outputs/qwen")
ANNOTATIONS_FILE = os.path.join(ROOT_DIR, "data/YouCookII/annotations/youcookii_annotations_trainval.json")

# Auto-discover videos and create sampled file if it doesn't exist
discover_and_create_sampled_file(BASE_FOLDER, SAMPLED_FILE)

if not hf_key:
    raise RuntimeError("HUGGINGFACE_KEY not found in .env file.")
login(token=hf_key)

aggregator, processor, model = initialize_models(
    aggregator_model_name=AGGREGATOR_MODEL,
    cot_model_name=COT_MODEL,
    agg_gpu=0,
    cot_gpu=0
)

Sampled file already exists: /Users/eddisonpham/Projects/DynaStride-Dynamic-Stride-Windowing-with-MMCoT-for-Multi-Scene-Captioning/constants/sampled_videos.txt


Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 40.17it/s]
Device set to use cpu
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
Loading checkpoint shards: 100%|██████████| 5/5 [00:51<00:00, 10.29s/it]


: 

In [ ]:
run_pipeline(
    base_folder=BASE_FOLDER,
    sampled_file=SAMPLED_FILE,
    output_folder=OUTPUT_FOLDER,
    annotations_file=ANNOTATIONS_FILE,
    aggregator=aggregator,
    processor=processor,
    model=model,
    COT_MODEL_ID=COT_MODEL
)

Found 223 videos to process.


Processing sampled videos:   0%|          | 0/223 [00:00<?, ?it/s]


--- [1/223] Processing video path: 101/10dZTHlkb8w
Processing frames in folder: /Users/eddisonpham/Projects/DynaStride-Dynamic-Stride-Windowing-with-MMCoT-for-Multi-Scene-Captioning/data/YouCookII/raw_videos/validation/101/10dZTHlkb8w
video_folder: /Users/eddisonpham/Projects/DynaStride-Dynamic-Stride-Windowing-with-MMCoT-for-Multi-Scene-Captioning/data/YouCookII/raw_videos/validation/101/10dZTHlkb8w, d: 10dZTHlkb8w_6_frames
video_folder: /Users/eddisonpham/Projects/DynaStride-Dynamic-Stride-Windowing-with-MMCoT-for-Multi-Scene-Captioning/data/YouCookII/raw_videos/validation/101/10dZTHlkb8w, d: 10dZTHlkb8w_8_frames
video_folder: /Users/eddisonpham/Projects/DynaStride-Dynamic-Stride-Windowing-with-MMCoT-for-Multi-Scene-Captioning/data/YouCookII/raw_videos/validation/101/10dZTHlkb8w, d: 10dZTHlkb8w_4_frames
video_folder: /Users/eddisonpham/Projects/DynaStride-Dynamic-Stride-Windowing-with-MMCoT-for-Multi-Scene-Captioning/data/YouCookII/raw_videos/validation/101/10dZTHlkb8w, d: 10dZTHlkb

scene_folder: 10dZTHlkb8w_6_frames
